In [2]:
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only!'}")

PyTorch version: 2.10.0+cu128
GPU available: True
Device: Tesla T4


In [3]:
!pip install scikit-learn seaborn -q

In [4]:
# ── Environment setup ─────────────────────────────────────────────
import torch
print(f"PyTorch: {torch.__version__}")
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Mount Google Drive for saving checkpoints and plots
from google.colab import drive
drive.mount('/content/drive')

# Create project folder in Drive
import os
SAVE_DIR = '/content/drive/MyDrive/DeepLearning_Project'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f"Save directory: {SAVE_DIR}")

PyTorch: 2.10.0+cu128
GPU available: True
GPU: Tesla T4
Mounted at /content/drive
Save directory: /content/drive/MyDrive/DeepLearning_Project


In [5]:
# ── Imports ───────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, random_split

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import time
from sklearn.metrics import classification_report, confusion_matrix,\
    accuracy_score, precision_score, recall_score, f1_score
import seaborn as sns

print("All imports successful")

All imports successful


In [6]:
# ── Config ────────────────────────────────────────────────────────
BATCH_SIZE  = 128       # larger batch = faster on GPU
VAL_SPLIT   = 0.1
NUM_WORKERS = 2         # safe on Colab
SEED        = 42

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2470, 0.2435, 0.2616)

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [7]:
# ── Transforms ────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
])

# ResNet transforms — resize to 224×224 (ImageNet standard)
resnet_train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(224, padding=16),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

resnet_eval_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Transforms ready")

Transforms ready


In [8]:
# ── Load & Split ──────────────────────────────────────────────────
full_train = torchvision.datasets.CIFAR10(root='./data', train=True,
                                           download=True,
                                           transform=train_transform)
full_train_vis = torchvision.datasets.CIFAR10(root='./data', train=True,
                                               download=False,
                                               transform=eval_transform)
test_set = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=False,
                                         transform=eval_transform)

val_size   = int(len(full_train) * VAL_SPLIT)
train_size = len(full_train) - val_size
train_set, val_set = random_split(full_train, [train_size, val_size],
                                  generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE,
                          shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {train_size} | Val: {val_size} | Test: {len(test_set)}")

100%|██████████| 170M/170M [00:03<00:00, 47.3MB/s]


Train: 45000 | Val: 5000 | Test: 10000


In [9]:
# ── Visualizations ────────────────────────────────────────────────
def denormalize(tensor, mean=CIFAR10_MEAN, std=CIFAR10_STD):
    t = tensor.clone()
    for c, (m, s) in enumerate(zip(mean, std)):
        t[c] = t[c] * s + m
    return t.clamp(0, 1)

def show_samples(dataset, classes, n=10,
                 save_path=f'{SAVE_DIR}/sample_images.png'):
    indices = np.random.choice(len(dataset), n, replace=False)
    fig, axes = plt.subplots(2, n // 2, figsize=(14, 6))
    fig.suptitle('CIFAR-10 Sample Images', fontsize=14, fontweight='bold')
    for i, idx in enumerate(indices):
        img, label = dataset[idx]
        img = denormalize(img).permute(1, 2, 0).numpy()
        ax = axes[i // (n // 2)][i % (n // 2)]
        ax.imshow(img)
        ax.set_title(classes[label], fontsize=10)
        ax.axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved → {save_path}")

def show_class_distribution(dataset, classes,
                             save_path=f'{SAVE_DIR}/class_distribution.png'):
    counts = np.zeros(len(classes), dtype=int)
    for _, label in dataset:
        counts[label] += 1
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(classes, counts, color='steelblue', edgecolor='black')
    ax.set_title('Class Distribution (Training Set)', fontweight='bold')
    ax.set_ylabel('Number of Images')
    plt.xticks(rotation=15)
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 50,
                str(count), ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved → {save_path}")

show_samples(full_train_vis, CLASSES)
show_class_distribution(full_train_vis, CLASSES)

Saved → /content/drive/MyDrive/DeepLearning_Project/sample_images.png
Saved → /content/drive/MyDrive/DeepLearning_Project/class_distribution.png


In [10]:
# ── CNN Architecture ──────────────────────────────────────────────
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout2d(dropout)
        )
    def forward(self, x):
        return self.block(x)

class CNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            ConvBlock(3,   32,  dropout=0.2),
            ConvBlock(32,  64,  dropout=0.3),
            ConvBlock(64,  128, dropout=0.4),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = CNN(num_classes=10).to(device)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")

model.eval()
with torch.no_grad():
    dummy = torch.zeros(1, 3, 32, 32).to(device)
    print(f"Output shape: {model(dummy).shape}")
model.train()

Total trainable parameters: 1,405,226
Output shape: torch.Size([1, 10])


CNN(
  (features): Sequential(
    (0): ConvBlock(
      (block): Sequential(
        (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (5): ReLU(inplace=True)
        (6): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
        (7): Dropout2d(p=0.2, inplace=False)
      )
    )
    (1): ConvBlock(
      (block): Sequential(
        (0): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU(inplace=True)
        (3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
        (4): BatchNorm2d(64, eps=1e-05

In [11]:
# ── Train & Validate Functions ────────────────────────────────────
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total   += labels.size(0)
    return running_loss / total, correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            running_loss += loss.item() * images.size(0)
            _, predicted  = outputs.max(1)
            correct += predicted.eq(labels).sum().item()
            total   += labels.size(0)
    return running_loss / total, correct / total

In [12]:
# ── Train CNN ─────────────────────────────────────────────────────
EPOCHS        = 50
LEARNING_RATE = 0.001
PATIENCE      = 7

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                             lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3)

cnn_history    = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
best_val_loss  = float('inf')
patience_count = 0
best_weights   = None

print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} "
      f"{'Val Loss':>10} {'Val Acc':>9} {'Time':>7}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch(model, train_loader,
                                            optimizer, criterion, device)
    val_loss, val_acc     = validate(model, val_loader, criterion, device)
    scheduler.step(val_loss)

    cnn_history['train_loss'].append(train_loss)
    cnn_history['val_loss'].append(val_loss)
    cnn_history['train_acc'].append(train_acc)
    cnn_history['val_acc'].append(val_acc)

    elapsed = time.time() - start
    print(f"{epoch:>6} {train_loss:>11.4f} {train_acc:>10.4f} "
          f"{val_loss:>10.4f} {val_acc:>9.4f} {elapsed:>6.1f}s")

    if val_loss < best_val_loss:
        best_val_loss  = val_loss
        patience_count = 0
        best_weights   = {k: v.clone() for k, v in model.state_dict().items()}
        # Save to Drive immediately
        torch.save(best_weights, f'{SAVE_DIR}/cnn_best_weights.pth')
        print(f"         ✓ Best model saved (val_loss={best_val_loss:.4f})")
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

model.load_state_dict(best_weights)
torch.save({'model_state_dict': best_weights,
            'history': cnn_history,
            'best_val_loss': best_val_loss},
           f'{SAVE_DIR}/cnn_checkpoint.pth')
print(f"\nCNN training complete. Best val_loss: {best_val_loss:.4f}")

 Epoch  Train Loss  Train Acc   Val Loss   Val Acc    Time
------------------------------------------------------------
     1      1.8063     0.3246     1.4650    0.4614   34.2s
         ✓ Best model saved (val_loss=1.4650)
     2      1.4830     0.4606     1.2382    0.5578   33.2s
         ✓ Best model saved (val_loss=1.2382)
     3      1.3013     0.5304     1.0623    0.6232   33.5s
         ✓ Best model saved (val_loss=1.0623)
     4      1.1849     0.5768     0.9650    0.6498   33.8s
         ✓ Best model saved (val_loss=0.9650)
     5      1.0961     0.6110     0.8981    0.6830   34.7s
         ✓ Best model saved (val_loss=0.8981)
     6      1.0347     0.6351     0.8252    0.7074   49.3s
         ✓ Best model saved (val_loss=0.8252)
     7      0.9938     0.6517     0.7857    0.7210   33.5s
         ✓ Best model saved (val_loss=0.7857)
     8      0.9496     0.6668     0.7578    0.7290   33.8s
         ✓ Best model saved (val_loss=0.7578)
     9      0.9160     0.6827     0.7248

In [13]:
# ── CNN Training Curves ───────────────────────────────────────────
def plot_training_history(history, title='Training History',
                          save_path=None):
    epochs = range(1, len(history['train_loss']) + 1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(title, fontsize=14, fontweight='bold')

    ax1.plot(epochs, history['train_loss'], label='Train', color='steelblue')
    ax1.plot(epochs, history['val_loss'],   label='Val',   color='tomato')
    ax1.set_title('Loss'); ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss'); ax1.legend(); ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, history['train_acc'], label='Train', color='steelblue')
    ax2.plot(epochs, history['val_acc'],   label='Val',   color='tomato')
    ax2.set_title('Accuracy'); ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved → {save_path}")
    plt.show()
    plt.close()

plot_training_history(cnn_history, title='CNN Training History',
                      save_path=f'{SAVE_DIR}/cnn_training_curves.png')

Saved → /content/drive/MyDrive/DeepLearning_Project/cnn_training_curves.png


In [14]:
# ── Evaluate CNN ──────────────────────────────────────────────────
def evaluate_model(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)

def plot_confusion_matrix(y_true, y_pred, classes, title, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes, yticklabels=classes, ax=ax)
    ax.set_title(title, fontsize=14, fontweight='bold')
    ax.set_ylabel('True Label'); ax.set_xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f"Saved → {save_path}")

def plot_per_class_accuracy(y_true, y_pred, classes, title, save_path):
    cm  = confusion_matrix(y_true, y_pred)
    acc = cm.diagonal() / cm.sum(axis=1) * 100
    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(classes, acc, color='steelblue', edgecolor='black')
    ax.axhline(acc.mean(), color='tomato', linestyle='--',
               label=f'Mean: {acc.mean():.1f}%')
    ax.set_title(title, fontweight='bold')
    ax.set_ylabel('Accuracy (%)'); ax.set_ylim(0, 100); ax.legend()
    plt.xticks(rotation=15)
    for bar, a in zip(bars, acc):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 1,
                f'{a:.1f}%', ha='center', fontsize=9)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close()
    print(f"Saved → {save_path}")

y_true_cnn, y_pred_cnn = evaluate_model(model, test_loader, device)

print("=" * 60)
print("CUSTOM CNN — Classification Report")
print("=" * 60)
print(classification_report(y_true_cnn, y_pred_cnn, target_names=CLASSES))

plot_confusion_matrix(y_true_cnn, y_pred_cnn, CLASSES,
                      title='Custom CNN — Confusion Matrix',
                      save_path=f'{SAVE_DIR}/cnn_confusion_matrix.png')

plot_per_class_accuracy(y_true_cnn, y_pred_cnn, CLASSES,
                        title='Custom CNN — Per-Class Accuracy',
                        save_path=f'{SAVE_DIR}/cnn_per_class_accuracy.png')

CUSTOM CNN — Classification Report
              precision    recall  f1-score   support

    airplane       0.86      0.89      0.87      1000
  automobile       0.94      0.94      0.94      1000
        bird       0.85      0.78      0.81      1000
         cat       0.77      0.68      0.72      1000
        deer       0.85      0.88      0.87      1000
         dog       0.76      0.84      0.80      1000
        frog       0.90      0.91      0.90      1000
       horse       0.91      0.89      0.90      1000
        ship       0.93      0.92      0.92      1000
       truck       0.89      0.94      0.91      1000

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000

Saved → /content/drive/MyDrive/DeepLearning_Project/cnn_confusion_matrix.png
Saved → /content/drive/MyDrive/DeepLearning_Project/cnn_per_class_accuracy.png


In [15]:
# ── ResNet18 Transfer Learning ────────────────────────────────────
def build_resnet18(num_classes=10, freeze_backbone=True):
    resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    if freeze_backbone:
        for param in resnet.parameters():
            param.requires_grad = False
    resnet.fc = nn.Sequential(
        nn.Linear(512, 256),
        nn.ReLU(inplace=True),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return resnet

resnet_model = build_resnet18(num_classes=10, freeze_backbone=True).to(device)
total    = sum(p.numel() for p in resnet_model.parameters())
trainable = sum(p.numel() for p in resnet_model.parameters()
                if p.requires_grad)
print(f"Total params:     {total:,}")
print(f"Trainable params: {trainable:,}")

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 122MB/s]


Total params:     11,310,410
Trainable params: 133,898


In [16]:
# ── ResNet18 DataLoaders ──────────────────────────────────────────
rn_train = torchvision.datasets.CIFAR10(root='./data', train=True,
                                         download=False,
                                         transform=resnet_train_transform)
rn_test  = torchvision.datasets.CIFAR10(root='./data', train=False,
                                         download=False,
                                         transform=resnet_eval_transform)

rn_val_size   = int(len(rn_train) * VAL_SPLIT)
rn_train_size = len(rn_train) - rn_val_size
rn_train_set, rn_val_set = random_split(
    rn_train, [rn_train_size, rn_val_size],
    generator=torch.Generator().manual_seed(SEED))

rn_train_loader = DataLoader(rn_train_set, batch_size=BATCH_SIZE,
                              shuffle=True,  num_workers=NUM_WORKERS,
                              pin_memory=True)
rn_val_loader   = DataLoader(rn_val_set,   batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS,
                              pin_memory=True)
rn_test_loader  = DataLoader(rn_test,      batch_size=BATCH_SIZE,
                              shuffle=False, num_workers=NUM_WORKERS,
                              pin_memory=True)

print(f"ResNet loaders — Train: {rn_train_size} | "
      f"Val: {rn_val_size} | Test: {len(rn_test)}")

ResNet loaders — Train: 45000 | Val: 5000 | Test: 10000


In [17]:
# ── ResNet18 Phase 1 — Head Only ──────────────────────────────────
RN_EPOCHS  = 20
RN_PATIENCE = 5

rn_criterion = nn.CrossEntropyLoss()
rn_optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, resnet_model.parameters()),
    lr=0.001, weight_decay=1e-4)
rn_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    rn_optimizer, mode='min', factor=0.5, patience=3)

rn_history = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[]}
best_val_loss_rn  = float('inf')
patience_count_rn = 0
best_weights_rn   = None

print("Phase 1 — Training classifier head only")
print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} "
      f"{'Val Loss':>10} {'Val Acc':>9} {'Time':>7}")
print("-" * 60)

for epoch in range(1, RN_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch(resnet_model, rn_train_loader,
                                            rn_optimizer, rn_criterion, device)
    val_loss, val_acc     = validate(resnet_model, rn_val_loader,
                                     rn_criterion, device)
    rn_scheduler.step(val_loss)

    rn_history['train_loss'].append(train_loss)
    rn_history['val_loss'].append(val_loss)
    rn_history['train_acc'].append(train_acc)
    rn_history['val_acc'].append(val_acc)

    elapsed = time.time() - start
    print(f"{epoch:>6} {train_loss:>11.4f} {train_acc:>10.4f} "
          f"{val_loss:>10.4f} {val_acc:>9.4f} {elapsed:>6.1f}s")

    if val_loss < best_val_loss_rn:
        best_val_loss_rn  = val_loss
        patience_count_rn = 0
        best_weights_rn   = {k: v.clone()
                              for k, v in resnet_model.state_dict().items()}
        torch.save(best_weights_rn,
                   f'{SAVE_DIR}/resnet18_phase1_best.pth')
        print(f"         ✓ Best model saved (val_loss={best_val_loss_rn:.4f})")
    else:
        patience_count_rn += 1
        if patience_count_rn >= RN_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

resnet_model.load_state_dict(best_weights_rn)
print(f"\nPhase 1 complete. Best val_loss: {best_val_loss_rn:.4f}")

Phase 1 — Training classifier head only
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc    Time
------------------------------------------------------------
     1      0.9174     0.6900     0.6388    0.7718  183.5s
         ✓ Best model saved (val_loss=0.6388)
     2      0.7051     0.7604     0.6180    0.7852  181.1s
         ✓ Best model saved (val_loss=0.6180)
     3      0.6740     0.7678     0.5925    0.7956  183.4s
         ✓ Best model saved (val_loss=0.5925)
     4      0.6449     0.7773     0.5816    0.7958  185.9s
         ✓ Best model saved (val_loss=0.5816)
     5      0.6371     0.7794     0.5485    0.8112  182.4s
         ✓ Best model saved (val_loss=0.5485)
     6      0.6285     0.7815     0.5431    0.8044  183.1s
         ✓ Best model saved (val_loss=0.5431)
     7      0.6227     0.7828     0.5559    0.8034  182.9s
     8      0.6197     0.7852     0.5489    0.8112  181.4s
     9      0.6116     0.7874     0.5641    0.7982  182.1s
    10      0.6033     0.7905    

In [19]:
# ── ResNet18 Phase 2 — Full Fine-tuning ───────────────────────────
for param in resnet_model.parameters():
    param.requires_grad = True

rn_optimizer2 = torch.optim.Adam(resnet_model.parameters(),
                                  lr=1e-4, weight_decay=1e-4)
rn_scheduler2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    rn_optimizer2, mode='min', factor=0.5, patience=3)

best_val_loss_rn2  = float('inf')
patience_count_rn2 = 0

print("Phase 2 — Full fine-tuning (all layers unfrozen)")
print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} "
      f"{'Val Loss':>10} {'Val Acc':>9} {'Time':>7}")
print("-" * 60)

for epoch in range(1, RN_EPOCHS + 1):
    start = time.time()
    train_loss, train_acc = train_one_epoch(resnet_model, rn_train_loader,
                                            rn_optimizer2, rn_criterion, device)
    val_loss, val_acc     = validate(resnet_model, rn_val_loader,
                                     rn_criterion, device)
    rn_scheduler2.step(val_loss)

    rn_history['train_loss'].append(train_loss)
    rn_history['val_loss'].append(val_loss)
    rn_history['train_acc'].append(train_acc)
    rn_history['val_acc'].append(val_acc)

    elapsed = time.time() - start
    print(f"{epoch:>6} {train_loss:>11.4f} {train_acc:>10.4f} "
          f"{val_loss:>10.4f} {val_acc:>9.4f} {elapsed:>6.1f}s")

    if val_loss < best_val_loss_rn2:
        best_val_loss_rn2  = val_loss
        patience_count_rn2 = 0
        best_weights_rn    = {k: v.clone()
                               for k, v in resnet_model.state_dict().items()}
        torch.save(best_weights_rn,
                   f'{SAVE_DIR}/resnet18_phase2_best.pth')
        print(f"         ✓ Best model saved (val_loss={best_val_loss_rn2:.4f})")
    else:
        patience_count_rn2 += 1
        if patience_count_rn2 >= RN_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}")
            break

resnet_model.load_state_dict(best_weights_rn)
torch.save({'model_state_dict': best_weights_rn,
            'history': rn_history,
            'best_val_loss': best_val_loss_rn2},
           f'{SAVE_DIR}/resnet18_checkpoint.pth')
print(f"\nPhase 2 complete. Best val_loss: {best_val_loss_rn2:.4f}")

Phase 2 — Full fine-tuning (all layers unfrozen)
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc    Time
------------------------------------------------------------
     1      0.3022     0.8980     0.2100    0.9306  211.9s
         ✓ Best model saved (val_loss=0.2100)
     2      0.1689     0.9424     0.1923    0.9316  211.2s
         ✓ Best model saved (val_loss=0.1923)
     3      0.1277     0.9559     0.1723    0.9416  212.6s
         ✓ Best model saved (val_loss=0.1723)
     4      0.1022     0.9650     0.2001    0.9410  211.9s
     5      0.0834     0.9705     0.1621    0.9500  212.0s
         ✓ Best model saved (val_loss=0.1621)
     6      0.0634     0.9785     0.1637    0.9480  213.7s
     7      0.0660     0.9770     0.1617    0.9504  212.4s
         ✓ Best model saved (val_loss=0.1617)
     8      0.0583     0.9799     0.1543    0.9520  212.3s
         ✓ Best model saved (val_loss=0.1543)
     9      0.0523     0.9817     0.1832    0.9470  213.3s
    10      0.0479     0

In [20]:
# ── Evaluate ResNet18 ─────────────────────────────────────────────
plot_training_history(rn_history, title='ResNet18 Training History',
                      save_path=f'{SAVE_DIR}/resnet18_training_curves.png')

y_true_rn, y_pred_rn = evaluate_model(resnet_model, rn_test_loader, device)

print("=" * 60)
print("RESNET18 — Classification Report")
print("=" * 60)
print(classification_report(y_true_rn, y_pred_rn, target_names=CLASSES))

plot_confusion_matrix(y_true_rn, y_pred_rn, CLASSES,
                      title='ResNet18 — Confusion Matrix',
                      save_path=f'{SAVE_DIR}/resnet18_confusion_matrix.png')

plot_per_class_accuracy(y_true_rn, y_pred_rn, CLASSES,
                        title='ResNet18 — Per-Class Accuracy',
                        save_path=f'{SAVE_DIR}/resnet18_per_class_accuracy.png')

Saved → /content/drive/MyDrive/DeepLearning_Project/resnet18_training_curves.png
RESNET18 — Classification Report
              precision    recall  f1-score   support

    airplane       0.96      0.97      0.97      1000
  automobile       0.97      0.97      0.97      1000
        bird       0.98      0.93      0.95      1000
         cat       0.92      0.89      0.91      1000
        deer       0.95      0.98      0.96      1000
         dog       0.91      0.94      0.93      1000
        frog       0.98      0.98      0.98      1000
       horse       0.98      0.97      0.98      1000
        ship       0.97      0.98      0.98      1000
       truck       0.96      0.97      0.97      1000

    accuracy                           0.96     10000
   macro avg       0.96      0.96      0.96     10000
weighted avg       0.96      0.96      0.96     10000

Saved → /content/drive/MyDrive/DeepLearning_Project/resnet18_confusion_matrix.png
Saved → /content/drive/MyDrive/DeepLearning_P

In [21]:
# ── Final Comparison ──────────────────────────────────────────────
def get_metrics(y_true, y_pred):
    return {
        'Accuracy':  accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred, average='weighted'),
        'Recall':    recall_score(y_true, y_pred, average='weighted'),
        'F1-Score':  f1_score(y_true, y_pred, average='weighted'),
    }

cnn_metrics    = get_metrics(y_true_cnn, y_pred_cnn)
resnet_metrics = get_metrics(y_true_rn,  y_pred_rn)

print("=" * 52)
print(f"{'Metric':<12} {'Custom CNN':>14} {'ResNet18':>14}")
print("=" * 52)
for metric in cnn_metrics:
    print(f"{metric:<12} {cnn_metrics[metric]:>14.4f} "
          f"{resnet_metrics[metric]:>14.4f}")
print("=" * 52)

# Bar chart comparison
metrics   = list(cnn_metrics.keys())
cnn_vals  = [cnn_metrics[m]    for m in metrics]
rn_vals   = [resnet_metrics[m] for m in metrics]
x = np.arange(len(metrics)); width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
b1 = ax.bar(x - width/2, cnn_vals,  width, label='Custom CNN',
            color='steelblue', edgecolor='black')
b2 = ax.bar(x + width/2, rn_vals,   width, label='ResNet18',
            color='tomato',    edgecolor='black')
ax.set_title('Custom CNN vs ResNet18 — Performance Comparison',
             fontweight='bold')
ax.set_ylabel('Score'); ax.set_xticks(x)
ax.set_xticklabels(metrics); ax.set_ylim(0, 1.1)
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
for bar in list(b1) + list(b2):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print(f"Saved → {SAVE_DIR}/model_comparison.png")

Metric           Custom CNN       ResNet18
Accuracy             0.8662         0.9583
Precision            0.8659         0.9584
Recall               0.8662         0.9583
F1-Score             0.8653         0.9582
Saved → /content/drive/MyDrive/DeepLearning_Project/model_comparison.png


In [23]:
# Test Model

# ── Predict your own images ───────────────────────────────────────
from google.colab import files
from PIL import Image
import torch.nn.functional as F
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

def predict_image(image_path, model, device, classes):
    transform = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD)
    ])
    img_original = Image.open(image_path).convert('RGB')
    tensor = transform(img_original).unsqueeze(0).to(device)
    model.eval()
    with torch.no_grad():
        logits = model(tensor)
        probs  = F.softmax(logits, dim=1)[0]
    top3_probs, top3_idx = probs.topk(3)
    top3_classes = [classes[i] for i in top3_idx.cpu()]
    top3_values  = top3_probs.cpu().numpy() * 100
    return img_original, top3_classes, top3_values

# ── Upload ────────────────────────────────────────────────────────
print("Select your images — you can select multiple files at once")
uploaded = files.upload()

# ── Run & Plot ────────────────────────────────────────────────────
n   = len(uploaded)
fig = plt.figure(figsize=(14, n * 3.5))
gs  = gridspec.GridSpec(n, 3, figure=fig, hspace=0.5, wspace=0.3)

for row, filename in enumerate(uploaded.keys()):
    with open(filename, 'wb') as f:
        f.write(uploaded[filename])

    img, cnn_classes, cnn_probs = predict_image(filename, model,        device, CLASSES)
    _,   rn_classes,  rn_probs  = predict_image(filename, resnet_model, device, CLASSES)

    # Original image
    ax_img = fig.add_subplot(gs[row, 0])
    ax_img.imshow(img)
    ax_img.axis('off')
    ax_img.set_title(filename[:20], fontsize=9, fontweight='bold')

    # Custom CNN bar chart
    ax_cnn = fig.add_subplot(gs[row, 1])
    colors_cnn = ['#4472A4', '#7BAFD4', '#B8D4E8']
    ax_cnn.barh(cnn_classes[::-1], cnn_probs[::-1], color=colors_cnn[::-1])
    ax_cnn.set_xlim(0, 105)
    ax_cnn.set_title('Custom CNN', fontsize=9, fontweight='bold')
    ax_cnn.set_xlabel('Confidence (%)', fontsize=8)
    for i, (cls, prob) in enumerate(zip(cnn_classes[::-1], cnn_probs[::-1])):
        ax_cnn.text(prob + 1, i, f'{prob:.1f}%', va='center', fontsize=8)

    # ResNet18 bar chart
    ax_rn = fig.add_subplot(gs[row, 2])
    colors_rn = ['#2D8A5E', '#7BC4A0', '#B8E0CC']
    ax_rn.barh(rn_classes[::-1], rn_probs[::-1], color=colors_rn[::-1])
    ax_rn.set_xlim(0, 105)
    ax_rn.set_title('ResNet18', fontsize=9, fontweight='bold')
    ax_rn.set_xlabel('Confidence (%)', fontsize=8)
    for i, (cls, prob) in enumerate(zip(rn_classes[::-1], rn_probs[::-1])):
        ax_rn.text(prob + 1, i, f'{prob:.1f}%', va='center', fontsize=8)

fig.suptitle('Custom CNN vs ResNet18 — Your Images',
             fontsize=13, fontweight='bold', y=1.01)
plt.savefig(f'{SAVE_DIR}/my_image_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved → {SAVE_DIR}/my_image_predictions.png")

Select your images — you can select multiple files at once


Saving test_image_1.png to test_image_1 (1).png
Saving test_image_2.png to test_image_2 (1).png
Saving test_image_3.png to test_image_3 (1).png
Saving test_image_4.png to test_image_4 (1).png
Saving test_image_5.png to test_image_5 (1).png
Saving test_image_6.png to test_image_6 (1).png
Saved → /content/drive/MyDrive/DeepLearning_Project/my_image_predictions.png
